# 05 — Service Gaps

This notebook examines the operational problems that groups have actually experienced: service observations, wall-time terminations, checkpoint abandonment, and scaling behaviour. These are factual reports that the committee uses to identify where current service provision is insufficient.

**Run `00_setup.ipynb` first.**

In [ ]:
import sys
sys.path.insert(0, '/content')
sys.path.insert(0, '')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sheets_client import load_sheets, explode_semicolons

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

dfs = load_sheets()
svc = dfs.get('ServiceObservations', pd.DataFrame())
wtt = dfs.get('WallTimeTerminations', pd.DataFrame())
chk = dfs.get('CheckpointInfo', pd.DataFrame())
scl = dfs.get('ScalingInfo', pd.DataFrame())

print(f"ServiceObservations: {len(svc)}")
print(f"WallTimeTerminations: {len(wtt)}")
print(f"CheckpointInfo: {len(chk)}")
print(f"ScalingInfo: {len(scl)}")

---
## Chart 1 — Problems Experienced

Counts of problem types reported in ServiceObservations. Multiple problems per submission are split and counted individually.

**What to look for:** Which service problems are most common? This gives a direct view of which QoS dimensions are most needed.

In [ ]:
PROBLEM_LABELS = {
    'wall_time_termination': 'Jobs terminated by wall-time limit',
    'excessive_queue_wait': 'Excessive queue waiting time',
    'insufficient_node_count': 'Insufficient node count',
    'insufficient_memory': 'Insufficient memory',
    'insufficient_gpu_access': 'Insufficient GPU access',
    'insufficient_total_capacity': 'Insufficient total capacity',
    'storage': 'Insufficient storage',
    'software': 'Software / environment issues',
    'network': 'Network / interconnect issues',
    'none': 'No problems experienced',
    'other': 'Other',
}

if svc.empty or 'problems_experienced' not in svc.columns:
    print("No problems_experienced data in ServiceObservations.")
else:
    probs = explode_semicolons(svc, 'problems_experienced')
    probs = probs.map(lambda x: PROBLEM_LABELS.get(x.strip(), x.strip()))
    prob_counts = probs.value_counts()

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(prob_counts.index[::-1], prob_counts.values[::-1],
                   color=sns.color_palette('muted')[0])
    for bar, val in zip(bars, prob_counts.values[::-1]):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
                str(val), va='center', ha='left', fontsize=9)
    ax.set_xlabel('Number of submissions reporting this problem')
    ax.set_title('Service Problems Experienced (Section N)', fontsize=13)
    plt.tight_layout()
    plt.show()

---
## Chart 2 — Workload Characteristics Flagged

Counts of workload characteristics self-reported in Section N (split by semicolons).

**What to look for:** Which workload characteristics are most commonly identified? This helps the committee understand what service requirements groups themselves associate with their workloads.

In [ ]:
if svc.empty or 'workload_characteristics' not in svc.columns:
    print("No workload_characteristics data in ServiceObservations.")
else:
    chars = explode_semicolons(svc, 'workload_characteristics')
    char_counts = chars.str.strip().replace('', pd.NA).dropna().value_counts()

    if char_counts.empty:
        print("No workload characteristic values found.")
    else:
        fig, ax = plt.subplots(figsize=(10, 5))
        bars = ax.barh(char_counts.index[::-1], char_counts.values[::-1],
                       color=sns.color_palette('muted')[2])
        for bar, val in zip(bars, char_counts.values[::-1]):
            ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
                    str(val), va='center', ha='left', fontsize=9)
        ax.set_xlabel('Number of submissions')
        ax.set_title('Workload Characteristics Flagged in Section N', fontsize=13)
        plt.tight_layout()
        plt.show()

---
## Chart 3 — Wall-Time Termination Frequency

Distribution of termination frequency among groups that report having had jobs terminated by a wall-time limit. Only rows where has_been_terminated = yes are included.

**What to look for:** Is wall-time termination a rare event or a routine one? Frequent terminations suggest systematic mismatch between workload requirements and available wall-time limits.

In [ ]:
FREQ_LABELS = {
    'once': 'Once only',
    'rarely': 'Rarely (< once per month)',
    'occasionally': 'Occasionally (monthly)',
    'regularly': 'Regularly (weekly)',
    'frequently': 'Frequently (multiple per week)',
    'very_frequently': 'Very frequently (daily)',
    'dont_know': "Don't know",
}
FREQ_ORDER = list(FREQ_LABELS.values())

if wtt.empty or 'frequency' not in wtt.columns:
    print("No wall-time termination frequency data available.")
else:
    terminated_col = next((c for c in ['has_been_terminated', 'terminated'] if c in wtt.columns), None)
    if terminated_col:
        df_wtt = wtt[wtt[terminated_col].str.strip().str.lower() == 'yes'].copy()
    else:
        df_wtt = wtt.copy()

    freq = (
        df_wtt['frequency']
        .dropna().str.strip()
        .loc[lambda s: s != '']
        .map(lambda x: FREQ_LABELS.get(x, x))
        .value_counts()
    )
    freq = freq.reindex([c for c in FREQ_ORDER if c in freq.index])

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.bar(freq.index, freq.values, color='tomato')
    for bar, val in zip(bars, freq.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05, str(val),
                ha='center', va='bottom', fontsize=9)
    ax.set_xlabel('Frequency of wall-time termination')
    ax.set_ylabel('Number of submissions')
    ax.set_title('Wall-Time Termination Frequency\n(terminated = yes submissions only)', fontsize=12)
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.show()
    print(f"Submissions with termination events: {len(df_wtt)}")

---
## Chart 4 — Where Terminated Calculations Eventually Completed

Shows whether calculations that were killed by a wall-time limit eventually completed elsewhere (different system, larger allocation, checkpoint restart, etc.).

**What to look for:** Calculations that completed elsewhere confirm the workload is technically feasible — the wall-time limit was the constraint, not the workload itself.

In [ ]:
COMPLETED_LABELS = {
    'yes_same_system': 'Yes — same system, larger allocation',
    'yes_different_system': 'Yes — different system',
    'yes_checkpoint': 'Yes — via checkpoint/restart',
    'no': 'No — not completed',
    'ongoing': 'Still in progress',
    'dont_know': "Don't know",
}

if wtt.empty or 'completed_anywhere' not in wtt.columns:
    print("No completed_anywhere data in WallTimeTerminations.")
else:
    comp = (
        wtt['completed_anywhere']
        .dropna().str.strip()
        .loc[lambda s: s != '']
        .map(lambda x: COMPLETED_LABELS.get(x, x))
        .value_counts()
    )

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.barh(comp.index[::-1], comp.values[::-1],
                   color=sns.color_palette('muted')[1])
    for bar, val in zip(bars, comp.values[::-1]):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
                str(val), va='center', ha='left', fontsize=9)
    ax.set_xlabel('Number of termination records')
    ax.set_title('Completion Status After Wall-Time Termination', fontsize=13)
    plt.tight_layout()
    plt.show()

---
## Chart 5 — Terminated CPU-Hours vs Completed CPU-Hours (scatter)

Each point is a WallTimeTermination record. X = CPU-hours consumed at the point of termination; Y = CPU-hours for the successfully completed run (if available). Points above the diagonal indicate the completion required more resources.

**What to look for:** The gap between terminated and completed CPU-hours shows the resource overhead caused by inadequate wall-time limits.

In [ ]:
cols_needed = {'terminated_cpu_hours', 'completed_cpu_hours'}

if wtt.empty or not cols_needed.issubset(wtt.columns):
    print("terminated_cpu_hours or completed_cpu_hours columns not found in WallTimeTerminations.")
else:
    df_sc = wtt[['terminated_cpu_hours', 'completed_cpu_hours']].copy()
    df_sc['terminated_cpu_hours'] = pd.to_numeric(df_sc['terminated_cpu_hours'], errors='coerce')
    df_sc['completed_cpu_hours'] = pd.to_numeric(df_sc['completed_cpu_hours'], errors='coerce')
    df_sc = df_sc.dropna()
    df_sc = df_sc[(df_sc['terminated_cpu_hours'] > 0) & (df_sc['completed_cpu_hours'] > 0)]

    if df_sc.empty:
        print("No paired terminated/completed CPU-hour records found.")
    else:
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.scatter(
            df_sc['terminated_cpu_hours'],
            df_sc['completed_cpu_hours'],
            alpha=0.7, s=60, color=sns.color_palette('muted')[3]
        )
        lims = [
            min(df_sc['terminated_cpu_hours'].min(), df_sc['completed_cpu_hours'].min()) * 0.8,
            max(df_sc['terminated_cpu_hours'].max(), df_sc['completed_cpu_hours'].max()) * 1.2,
        ]
        ax.plot(lims, lims, 'k--', linewidth=0.8, label='Equal resources')
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlabel('CPU-hours at termination')
        ax.set_ylabel('CPU-hours for completed run')
        ax.set_title('CPU-Hours: Terminated Run vs Completed Run', fontsize=13)
        ax.legend(fontsize=8)
        ax.grid(True, which='both', alpha=0.3)
        plt.tight_layout()
        plt.show()
        print(f"Points: {len(df_sc)}")

---
## Chart 6 — Checkpoint Abandonment Reasons

Distribution of reasons given for abandoning checkpoint/restart (only for CheckpointInfo records where checkpoint_abandoned = yes).

**What to look for:** What barriers prevent groups from using checkpointing? Software limitations are beyond the user's control; configuration issues may be addressable with technical support.

In [ ]:
ABANDON_LABELS = {
    'not_supported': 'Not supported by software',
    'too_slow': 'Too slow / high overhead',
    'unreliable': 'Unreliable / corrupted checkpoints',
    'complicated': 'Too complicated to set up',
    'storage': 'Storage constraints',
    'not_needed': 'Not needed in practice',
    'dont_know': "Don't know",
    'other': 'Other',
}

if chk.empty or 'abandoned_reason' not in chk.columns:
    print("No checkpoint abandonment reason data available.")
else:
    abandoned_col = next((c for c in ['checkpoint_abandoned', 'abandoned'] if c in chk.columns), None)
    if abandoned_col:
        df_ab = chk[chk[abandoned_col].str.strip().str.lower() == 'yes'].copy()
    else:
        df_ab = chk.copy()

    ab_reasons = (
        df_ab['abandoned_reason']
        .dropna().str.strip()
        .loc[lambda s: s != '']
        .map(lambda x: ABANDON_LABELS.get(x, x))
        .value_counts()
    )

    if ab_reasons.empty:
        print("No checkpoint abandonment records found.")
    else:
        fig, ax = plt.subplots(figsize=(9, 4))
        bars = ax.barh(ab_reasons.index[::-1], ab_reasons.values[::-1],
                       color='sandybrown')
        for bar, val in zip(bars, ab_reasons.values[::-1]):
            ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
                    str(val), va='center', ha='left', fontsize=9)
        ax.set_xlabel('Number of checkpoint entries')
        ax.set_title('Checkpoint Abandonment Reasons\n(checkpoint_abandoned = yes entries)', fontsize=12)
        plt.tight_layout()
        plt.show()

---
## Chart 7 — Scaling Behaviour Distribution

Distribution of parallel scaling behaviours reported in ScalingInfo.

**What to look for:** Strong scaling (linear or super-linear) confirms that node-count increases are productive. Poor or unknown scaling may indicate that increasing node count is not the right solution, and the workload may have different QoS requirements (e.g. longer wall time on fewer cores).

In [ ]:
SCALING_LABELS = {
    'linear': 'Linear (ideal)',
    'sublinear': 'Sub-linear (good)',
    'superlinear': 'Super-linear',
    'poor': 'Poor scaling',
    'flat': 'Flat / saturated',
    'independent': 'Embarrassingly parallel (independent jobs)',
    'not_tested': 'Not tested',
    'dont_know': "Don't know",
    'not_applicable': 'Not applicable',
}

if scl.empty or 'scaling_behaviour' not in scl.columns:
    print("No scaling behaviour data in ScalingInfo.")
else:
    sc_vals = (
        scl['scaling_behaviour']
        .dropna().str.strip()
        .loc[lambda s: s != '']
        .map(lambda x: SCALING_LABELS.get(x, x))
        .value_counts()
    )

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.barh(sc_vals.index[::-1], sc_vals.values[::-1],
                   color=sns.color_palette('muted')[4])
    for bar, val in zip(bars, sc_vals.values[::-1]):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
                str(val), va='center', ha='left', fontsize=9)
    ax.set_xlabel('Number of scaling records')
    ax.set_title('Scaling Behaviour Distribution (ScalingInfo)', fontsize=13)
    plt.tight_layout()
    plt.show()